# Week 10 Practice: RNNs and LSTMs

CSCI1409 Deep Learning · based on the RNN and LSTM lecture

## Instructions

- Work in this notebook with Python and PyTorch. You can use Google Colab.
- Show your code and its output. Be ready to explain every step.
- Write short answers to the explanation questions in the **Your answer** cells.

In [7]:
import torch
import torch.nn as nn

## Task 1: The RNN cell

An RNN reads a sequence one element at a time. After each element it updates its **hidden state** $h_t$, a vector that summarizes everything it has read so far:

$$h_t = \tanh\left(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b\right), \qquad h_0 = 0$$

The same $W_{xh}$, $W_{hh}$ and $b$ are used at every step.

### 1a) A tiny RNN by hand

Use an RNN where everything is a single number: $w_x = 1$, $w_h = 0.5$, $b = 0$, $h_0 = 0$, so

$$h_t = \tanh(1 \cdot x_t + 0.5 \cdot h_{t-1})$$

1. Compute $h_1, h_2, h_3$ on paper for the sequence $A = [1, 0, 0]$. Write the calculation for each step. You may use `torch.tanh` or a calculator for the tanh values.
2. Write a `for` loop in PyTorch that computes all hidden states. Run it on $A$ and on $B = [0, 0, 1]$.
3. $A$ and $B$ contain the same numbers. Why is $h_3$ different? What happens to the "1" from step 1 of $A$ as time goes on?

In [8]:
w_x, w_h, b = 1.0, 0.5, 0.0

A = torch.tensor([1., 0., 0.])
B = torch.tensor([0., 0., 1.])

def run_rnn(xs):
    h = torch.tensor(0.0)       
    states = []
    for t, x in enumerate(xs, start=1):
        h = torch.tanh(w_x * x + w_h * h + b)
        states.append(h.item())
        print(f"h_{t} = tanh({w_x}*{x.item():.0f} + {w_h}*h_{t-1}) = {h.item():.4f}")
    return states

print("Sequence A = [1, 0, 0]")
hA = run_rnn(A)
print("\nSequence B = [0, 0, 1]")
hB = run_rnn(B)

Sequence A = [1, 0, 0]
h_1 = tanh(1.0*1 + 0.5*h_0) = 0.7616
h_2 = tanh(1.0*0 + 0.5*h_1) = 0.3634
h_3 = tanh(1.0*0 + 0.5*h_2) = 0.1797

Sequence B = [0, 0, 1]
h_1 = tanh(1.0*0 + 0.5*h_0) = 0.0000
h_2 = tanh(1.0*0 + 0.5*h_1) = 0.0000
h_3 = tanh(1.0*1 + 0.5*h_2) = 0.7616


**Your answer:**

**1. By hand, sequence A = [1, 0, 0]:**

- $h_1 = \tanh(1\cdot 1 + 0.5\cdot 0) = \tanh(1) \approx 0.7616$
- $h_2 = \tanh(1\cdot 0 + 0.5\cdot 0.7616) = \tanh(0.3808) \approx 0.3634$
- $h_3 = \tanh(1\cdot 0 + 0.5\cdot 0.3634) = \tanh(0.1817) \approx 0.1797$

For B = [0, 0, 1]: $h_1 = \tanh(0) = 0$, $h_2 = \tanh(0) = 0$, $h_3 = \tanh(1) \approx 0.7616$.

**3.** The RNN is sensitive to **order**, not only to which numbers appear. In B the "1" arrives at the last step, so it enters $h_3$ directly. In A the "1" arrives at step 1 and afterwards only survives through the recurrent term $w_h h_{t-1}$: each step it is multiplied by $w_h = 0.5$ (and squashed by tanh), so its influence shrinks: 0.76 → 0.36 → 0.18. The RNN gradually "forgets" early inputs. This is the same effect that causes vanishing gradients and makes long-term dependencies hard for a plain RNN.

### 1b) `nn.RNN`: shapes and parameters

With input size $D$ and hidden size $H$, an `nn.RNN` layer has

$$\text{params} = H \cdot D + H \cdot H + 2H$$

($W_{xh}$ is $H \times D$, $W_{hh}$ is $H \times H$, and PyTorch keeps two bias vectors of size $H$.)

With `batch_first=True`, the input has shape `[batch, T, D]`. The layer returns two things: `out` (the hidden state at every step) and `h_n` (the last hidden state).

| Case | $D$ | $H$ | batch | $T$ |
|---|---|---|---|---|
| A | 1 | 4 | 2 | 5 |
| B | 8 | 16 | 4 | 10 |
| C | 8 | 16 | 4 | 100 |
| D | 3 | 32 | 1 | 7 |

1. Predict the shape of `out`, the shape of `h_n` and the number of parameters for each case.
2. Check with `nn.RNN(D, H, batch_first=True)`. Pass `torch.randn(batch, T, D)` through it and count parameters with `sum(p.numel() for p in rnn.parameters())`.
3. B and C have the same number of parameters, but C reads a sequence 10 times longer. Why does the count not change?

In [9]:
cases = {"A": (1, 4, 2, 5), "B": (8, 16, 4, 10), "C": (8, 16, 4, 100), "D": (3, 32, 1, 7)}

for name, (D, H, batch, T) in cases.items():
    # predictions
    pred_out = (batch, T, H)
    pred_hn = (1, batch, H)
    pred_params = H * D + H * H + 2 * H

    # check with PyTorch
    rnn = nn.RNN(D, H, batch_first=True)
    x = torch.randn(batch, T, D)
    out, h_n = rnn(x)
    n_params = sum(p.numel() for p in rnn.parameters())

    print(f"Case {name}: out {tuple(out.shape)} (pred {pred_out}), "
          f"h_n {tuple(h_n.shape)} (pred {pred_hn}), "
          f"params {n_params} (pred {pred_params})")
    assert tuple(out.shape) == pred_out and tuple(h_n.shape) == pred_hn and n_params == pred_params

Case A: out (2, 5, 4) (pred (2, 5, 4)), h_n (1, 2, 4) (pred (1, 2, 4)), params 28 (pred 28)
Case B: out (4, 10, 16) (pred (4, 10, 16)), h_n (1, 4, 16) (pred (1, 4, 16)), params 416 (pred 416)
Case C: out (4, 100, 16) (pred (4, 100, 16)), h_n (1, 4, 16) (pred (1, 4, 16)), params 416 (pred 416)
Case D: out (1, 7, 32) (pred (1, 7, 32)), h_n (1, 1, 32) (pred (1, 1, 32)), params 1184 (pred 1184)


**Your answer:**

**1. Predictions** (`out` = `[batch, T, H]`, `h_n` = `[num_layers, batch, H]` with 1 layer; note that `h_n` is *not* batch-first even with `batch_first=True`):

| Case | `out` | `h_n` | params |
|---|---|---|---|
| A | [2, 5, 4] | [1, 2, 4] | 4·1 + 4·4 + 2·4 = 28 |
| B | [4, 10, 16] | [1, 4, 16] | 16·8 + 16·16 + 2·16 = 416 |
| C | [4, 100, 16] | [1, 4, 16] | 416 |
| D | [1, 7, 32] | [1, 1, 32] | 32·3 + 32·32 + 2·32 = 1184 |

**3.** The same weights $W_{xh}$, $W_{hh}$ and $b$ are reused at every time step (weight sharing across time). The parameter count depends only on $D$ and $H$, not on $T$. A longer sequence just means the cell is applied more times; it changes the shape of `out` (more steps), not the number of parameters.

## Task 2: LSTM

An LSTM keeps two memories: the hidden state $h_t$ and the **cell state** $c_t$ (long-term memory). Three **gates** with values between 0 and 1 (from a sigmoid) control the cell state, and a candidate $g_t$ (between -1 and 1, from tanh) is the new information:

- forget gate $f_t$: how much of the old memory to keep,
- input gate $i_t$: how much of the new information to write,
- output gate $o_t$: how much of the memory to show as $h_t$.

$$c_t = f_t \cdot c_{t-1} + i_t \cdot g_t, \qquad h_t = o_t \cdot \tanh(c_t)$$

### 2a) One LSTM step by hand

The old cell state is $c_{t-1} = 2$. The gates are already computed:

| Case | $f_t$ | $i_t$ | $g_t$ | $o_t$ |
|---|---|---|---|---|
| A | 1 | 0 | 0.5 | 1 |
| B | 0 | 1 | 0.5 | 1 |
| C | 0.5 | 0.5 | -1 | 0.5 |
| D | 0.9 | 0.2 | 0.5 | 0.8 |

1. Compute $c_t$ and $h_t$ on paper for each case.
2. Check with PyTorch. Put each column in a tensor and compute all four cases at once.
3. In which case does the LSTM keep its memory exactly? In which case does it throw the old memory away and write new information? What would $o_t = 0$ mean?

In [10]:
c_prev = 2.0

f = torch.tensor([1.0, 0.0, 0.5, 0.9])
i = torch.tensor([0.0, 1.0, 0.5, 0.2])
g = torch.tensor([0.5, 0.5, -1.0, 0.5])
o = torch.tensor([1.0, 1.0, 0.5, 0.8])

c_t = f * c_prev + i * g
h_t = o * torch.tanh(c_t)

for name, c, h in zip("ABCD", c_t, h_t):
    print(f"Case {name}: c_t = {c.item():.4f}, h_t = {h.item():.4f}")

Case A: c_t = 2.0000, h_t = 0.9640
Case B: c_t = 0.5000, h_t = 0.4621
Case C: c_t = 0.5000, h_t = 0.2311
Case D: c_t = 1.9000, h_t = 0.7650


**Your answer:**

**1. By hand** ($c_t = f_t c_{t-1} + i_t g_t$, $h_t = o_t \tanh(c_t)$, $c_{t-1}=2$):

- A: $c_t = 1\cdot 2 + 0\cdot 0.5 = 2$, $h_t = 1\cdot\tanh(2) \approx 0.9640$
- B: $c_t = 0\cdot 2 + 1\cdot 0.5 = 0.5$, $h_t = 1\cdot\tanh(0.5) \approx 0.4621$
- C: $c_t = 0.5\cdot 2 + 0.5\cdot(-1) = 0.5$, $h_t = 0.5\cdot\tanh(0.5) \approx 0.2311$
- D: $c_t = 0.9\cdot 2 + 0.2\cdot 0.5 = 1.9$, $h_t = 0.8\cdot\tanh(1.9) \approx 0.7650$

**3.**
- **Case A** keeps the memory exactly: $f_t = 1$ copies the old cell state and $i_t = 0$ writes nothing new, so $c_t = c_{t-1} = 2$. This is why an LSTM can carry information over many steps.
- **Case B** throws the old memory away ($f_t = 0$) and writes only the new information ($i_t = 1$), so $c_t = g_t = 0.5$.
- **$o_t = 0$** would give $h_t = 0$: the memory is still stored in $c_t$, but it is hidden from the output and the next layer at this step. The LSTM can "remember silently" and reveal the information later.

### 2b) An LSTM for sequence classification

A common use: read a whole sequence and output one class (for example, the sentiment of a sentence). This is called **many-to-one**: we use only the last hidden state.

```python
lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 3)
```

1. Pass `x = torch.randn(4, 10, 8)` through `lstm`. It returns `out, (h_n, c_n)`. Print the three shapes.
2. Take the last time step with `out[:, -1, :]` and pass it through `fc`. What is the shape of the result and what does it mean? Check with `torch.allclose` that `out[:, -1, :]` equals `h_n[0]`.
3. Count the parameters of `lstm` and of `nn.RNN(8, 16)`. Why does the LSTM have exactly 4 times more?

In [11]:
torch.manual_seed(0)
lstm = nn.LSTM(input_size=8, hidden_size=16, batch_first=True)
fc = nn.Linear(16, 3)

# 1. shapes
x = torch.randn(4, 10, 8)
out, (h_n, c_n) = lstm(x)
print("out:", out.shape)    # [4, 10, 16]
print("h_n:", h_n.shape)    # [1, 4, 16]
print("c_n:", c_n.shape)    # [1, 4, 16]

# 2. many-to-one: last time step -> classifier
last = out[:, -1, :]        # [4, 16]
logits = fc(last)
print("logits:", logits.shape)   # [4, 3]
print("out[:, -1, :] == h_n[0]:", torch.allclose(last, h_n[0]))

# 3. parameter counts
rnn = nn.RNN(8, 16)
p_lstm = sum(p.numel() for p in lstm.parameters())
p_rnn = sum(p.numel() for p in rnn.parameters())
print("LSTM params:", p_lstm)    # 1664
print("RNN params:", p_rnn)      # 416
print("ratio:", p_lstm / p_rnn)  # 4.0

out: torch.Size([4, 10, 16])
h_n: torch.Size([1, 4, 16])
c_n: torch.Size([1, 4, 16])
logits: torch.Size([4, 3])
out[:, -1, :] == h_n[0]: True
LSTM params: 1664
RNN params: 416
ratio: 4.0


**Your answer:**

**1.** `out`: [4, 10, 16] (hidden state for each of the 10 steps), `h_n`: [1, 4, 16], `c_n`: [1, 4, 16] (last hidden and cell state, first dim = number of layers).

**2.** `fc(out[:, -1, :])` has shape **[4, 3]**: for each of the 4 sequences in the batch, 3 scores (logits), one per class. Applying softmax/argmax gives the predicted class for each sequence. `torch.allclose(out[:, -1, :], h_n[0])` returns `True`: for a one-layer, one-direction LSTM, the output at the last step *is* the final hidden state.

**3.** LSTM: 1664 parameters, RNN: 416. The RNN has one transformation $W_{xh}x_t + W_{hh}h_{t-1} + b$ with $16\cdot 8 + 16\cdot 16 + 2\cdot 16 = 416$ parameters. The LSTM computes **four** such transformations with the same structure: forget gate $f_t$, input gate $i_t$, output gate $o_t$ and candidate $g_t$, each with its own weights and biases. So it has $4 \cdot 416 = 1664$ parameters (PyTorch stores them stacked as `weight_ih_l0` of shape [64, 8], `weight_hh_l0` of shape [64, 16] and two biases of size 64).

## Stretch (optional): many-to-many

Some tasks need an output at **every** step, for example a tag for every word in a sentence. Apply `fc` to the whole `out` from 2b (not only the last step). What is the shape of the result, and what does each part mean?

In [12]:
y = fc(out)          # nn.Linear acts on the last dimension, so it is applied at every time step
print(y.shape)       # torch.Size([4, 10, 3])

torch.Size([4, 10, 3])


**Your answer:**

The result has shape **[4, 10, 3]**: 4 = sequences in the batch, 10 = time steps (e.g. words), 3 = class scores. `nn.Linear` works on the last dimension, so the same `fc` layer is applied to the hidden state at every step. `y[b, t, :]` are the 3 logits (e.g. tag scores) for word `t` of sentence `b`. This is many-to-many: one prediction per input element.